# Cross-validation & sanity-check lab

Two cheap gates that run before any expensive tuning:

- **Sanity battery** (`sanity.py`) -- ten mechanical checks on a trade
  list that catch the classic ways a backtest lies (outlier dependence,
  cost fragility, one-ticker/one-month concentration, degenerate R,
  time-stop-dominated exits). Seconds, no data fetch.
- **Blocked CV** (`crossval.py`) -- split history into contiguous blocks.
  Fold stability shows whether the edge lives everywhere or one lucky
  quarter; **regime segmentation** shows under which market conditions it
  holds (the macro-defense test); leave-one-block-out selection shows
  whether the best threshold is stable. (Purge/embargo at block borders;
  CV trains on future blocks, so it measures *stability*, not the
  deployment number -- that is the walk-forward lab.)

In [ ]:
from pathlib import Path
import subprocess, sys, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd() if (Path.cwd() / 'config.json').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from rhengine import viz
from rhengine.costs import default_cost_bps
viz.use_style()
CFG = json.load(open(ROOT / 'config.json'))
RESULTS = ROOT / 'tests' / 'results'; RESULTS.mkdir(parents=True, exist_ok=True)

# Full-market event-loop passes are slow; cap the universe for a live
# notebook run and raise it (or set 0) when you want the real number.
MAX_TICKERS  = 0    # 0 = full universe (was 150; full-market for real data density)
LOOKBACK     = 400
# Audit FC-2: resolved from config.json costs.round_trip_bps (30.0), not
# hardcoded. The old literal 5 is a mega-cap assumption; this universe
# floors at $500k/day and pooled breakeven is ~16 bps. Override per-run by
# assigning after this line, and say so when you report the number.
COST_BPS     = default_cost_bps(CFG)
REFRESH      = False   # True -> recompute cached CSVs even if present
print('root:', ROOT, '| data_source:', CFG.get('data_source'))

## 1 -- Sanity battery on the current config

One backtest over the capped universe (in-process, cached to CSV), then
ten mechanical checks. FAIL on any red check before trusting a single
downstream metric. Point `TRADES_CSV` at any trades CSV to audit it
instead (e.g. `tests/results/refined_trades.csv`).

In [ ]:
from rhengine.backtest import run_backtest_on
from rhengine.data_local import load_bars_any
from rhengine.sanity import run_battery, load_trades_csv
import csv as _csv

TRADES_CSV = RESULTS / 'cv_trades.csv'   # or point at an existing trades CSV
if REFRESH or not TRADES_CSV.exists():
    bars_map, spy_bars = load_bars_any(CFG, lookback_days=LOOKBACK, verbose=False)
    if MAX_TICKERS: bars_map = dict(list(bars_map.items())[:MAX_TICKERS])
    bt = run_backtest_on(CFG, bars_map, spy_bars, min_score=3, cost_bps=COST_BPS)
    if bt:
        with open(TRADES_CSV, 'w', newline='') as f:
            w = _csv.DictWriter(f, fieldnames=list(bt[0].keys())); w.writeheader(); w.writerows(bt)
    print('backtested', len(bt), 'trades ->', TRADES_CSV.name)
else:
    print('using cached', TRADES_CSV.name)

In [ ]:
from rhengine.sanity import load_trades_csv, run_battery
trades = load_trades_csv(TRADES_CSV)
rows = run_battery(trades)
for name, status, value, note in rows:
    print(f'[{status:<4}] {name:<22} {value}')
viz.check_table(rows, title=f'Sanity battery -- {TRADES_CSV.name} ({len(trades)} trades)')
plt.show()

The check_table above is the at-a-glance view; the printed rows carry the
numbers. `FAIL`/`WARN` rows name what to investigate.

## 2 -- Blocked fold stability

One backtest, trades bucketed into K contiguous date blocks. A strategy
positive in most blocks is robust; one monster block carrying the mean
is regime luck.

In [ ]:
STRATEGY = 'bollinger'   # None = auto-regime
FOLDS    = 5
EMBARGO  = 15            # calendar-day purge at block borders
from rhengine.crossval import evaluate_folds, stability_summary, cv_select
from rhengine.data_local import load_bars_any
bars_map, spy_bars = load_bars_any(CFG, lookback_days=LOOKBACK, verbose=False)
if MAX_TICKERS: bars_map = dict(list(bars_map.items())[:MAX_TICKERS])
frows, ftrades = evaluate_folds(CFG, bars_map, spy_bars, k=FOLDS,
        min_score=3, forced_strategy=STRATEGY, cost_bps=COST_BPS, embargo_days=EMBARGO)
fdf = pd.DataFrame([{ 'fold': r['fold'], 'window': f"{r['lo']}..{r['hi']}",
        'n': (r['stats'] or {}).get('n', 0),
        'avgR': (r['stats'] or {}).get('avg_r', np.nan),
        'PF': (r['stats'] or {}).get('profit_factor', np.nan)} for r in frows])
display(fdf)
print(stability_summary(frows))

In [ ]:
fig, ax = viz.bars_signed([f"f{r.fold} {r.window[:19]}" for r in fdf.itertuples()],
        fdf['avgR'].fillna(0).tolist(),
        title=f'{STRATEGY or "auto"} -- avg R per time block (consistency test)',
        xlabel='avg R', fmt='{:+.3f}')
plt.show()

## 3 -- Regime segmentation (the macro-defense test)

Time-block CV shows *when* the edge held; this shows under *which*
conditions -- bucket the same trades by SPY regime (uptrend/downtrend/
choppy) and by VIX bucket. An edge positive in only one regime is a
routing signal (run the strategy only there, or add a macro filter that
flattens elsewhere -- e.g. exit to cash on a 50/200 death cross). An edge
positive across regimes is broadly robust.

In [ ]:
from rhengine.crossval import regime_stability
rs = regime_stability(ftrades)   # reuse the fold backtest's trades
def _row(k, s): return {'bucket': k, 'n': (s or {}).get('n', 0),
        'win%': round((s or {}).get('win_rate', np.nan)*100, 1),
        'avgR': (s or {}).get('avg_r', np.nan),
        'PF': (s or {}).get('profit_factor', np.nan)}
reg_df = pd.DataFrame([_row(k, s) for k, s in rs['by_regime'].items()])
vix_df = pd.DataFrame([_row(k, s) for k, s in rs['by_vix'].items()])
print('SPY regime:'); display(reg_df)
print('VIX bucket:'); display(vix_df)
print(f"{rs['regimes_positive']}/{rs['regimes_scored']} regimes positive:",
      ', '.join(rs['positive_regimes']) or 'none')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
if len(reg_df):
    viz.bars_signed(reg_df['bucket'].tolist(), reg_df['avgR'].fillna(0).tolist(),
        title=f'{STRATEGY or "auto"} -- avg R by SPY regime', xlabel='avg R', ax=axes[0])
if len(vix_df):
    viz.bars_signed(vix_df['bucket'].tolist(), vix_df['avgR'].fillna(0).tolist(),
        title='avg R by VIX bucket', xlabel='avg R', ax=axes[1])
plt.tight_layout(); plt.show()

## 4 -- Leave-one-block-out selection stability (optional)

For each block, tune on the others and test on the held-out block. If the
chosen value jumps around every fold, the 'best' threshold is noise.

In [ ]:
PARAM = 'volMult'
RUN_SELECT = False   # flip on to run (slower: re-backtests every candidate x fold)
if RUN_SELECT:
    pooled, fold_log = cv_select(CFG, bars_map, spy_bars, STRATEGY, [PARAM],
            k=FOLDS, min_score=3, cost_bps=COST_BPS, embargo_days=EMBARGO)
    sdf = pd.DataFrame([{ 'fold': f['fold'], 'chosen': f['chosen'],
            'test_avgR': (f['test'] or {}).get('avg_r', np.nan),
            'test_n': (f['test'] or {}).get('n', 0)} for f in fold_log])
    display(sdf)
    print('distinct winners:', sdf['chosen'].nunique())
else:
    print('set RUN_SELECT = True to run the selection loop')

---
**Gate order:** sanity battery clean -> fold stability broadly positive
-> walk-forward pooled-OOS win. Only then Monte Carlo the refined trade
list for the risk distribution. A FAIL here means stop and fix, not tune.